# 4. Statistiques et régression linéaire

On utilise `simple-statistics` pour les statistiques descriptives et `ml-regression` pour la régression linéaire.

## 4.1 Charger les données

In [ ]:
import pl from "nodejs-polars";

// inferSchemaLength : Polars lit 1000 lignes pour deviner le type de chaque colonne
// (sinon "mpg" est pris pour un entier et la lecture échoue sur 17.5).
const df = pl.readCSV("../data/auto-mpg.csv", { inferSchemaLength: 1000 });
console.log(df.head(5).toString());
console.log("Dimensions :", df.shape);

## 4.2 Statistiques descriptives avec Polars

In [ ]:
// Résumé statistique des colonnes numériques
console.log(df.describe().toString());

// Moyenne et écart-type d'une colonne
const mpgMean = df.getColumn("mpg").mean();
const mpgStd = df.select(pl.col("mpg").std()).getColumn("mpg").toArray()[0];
console.log(`MPG moyen : ${mpgMean.toFixed(2)}`);
console.log(`Écart-type MPG : ${mpgStd.toFixed(2)}`);

## 4.3 Statistiques avec simple-statistics

In [ ]:
import * as ss from "simple-statistics";

const mpg = df.getColumn("mpg").toArray();
const weight = df.getColumn("weight").toArray();

console.log("Moyenne :", ss.mean(mpg).toFixed(2));
console.log("Médiane :", ss.median(mpg));
// Variance et écart-type d'échantillon (même calcul que Polars)
console.log("Variance :", ss.sampleVariance(mpg).toFixed(2));
console.log("Écart-type :", ss.sampleStandardDeviation(mpg).toFixed(2));
console.log("Corrélation poids/MPG :", ss.sampleCorrelation(weight, mpg).toFixed(4));

## 4.4 Régression linéaire simple avec simple-statistics

In [ ]:
// Préparer les données sous la forme [[x, y], ...]
const points = weight.map((w, i) => [w, mpg[i]]);

const regression = ss.linearRegression(points);
const slope = regression.m;
const intercept = regression.b;

console.log(`Pente (a) : ${slope.toFixed(6)}`);
console.log(`Ordonnée à l'origine (b) : ${intercept.toFixed(4)}`);
console.log(`Équation : MPG = ${slope.toFixed(6)} × weight + ${intercept.toFixed(4)}`);

// Coefficient de détermination R²
const r2 = ss.rSquared(points, (x) => slope * x + intercept);
console.log(`R² : ${r2.toFixed(4)}`);

// Prédictions
for (const w of [2000, 3000, 4000, 5000]) {
  console.log(`Poids ${w} lbs → MPG prédit : ${(slope * w + intercept).toFixed(2)}`);
}

## 4.5 Régression linéaire avec ml-regression

In [ ]:
import { SimpleLinearRegression } from "ml-regression";

const regression2 = new SimpleLinearRegression(weight, mpg);

console.log(`Pente : ${regression2.slope.toFixed(6)}`);
console.log(`Ordonnée : ${regression2.intercept.toFixed(4)}`);
console.log(`Prédiction pour 3000 lbs : ${regression2.predict(3000).toFixed(2)}`);

// Score R²
const score = regression2.score(weight, mpg); // { r, r2, chi2, rmsd }
console.log(`Score R² : ${score.r2.toFixed(4)}`);

## 4.6 Visualiser la régression avec Plotly

On reprend la fonction `plotly(data, layout)` du notebook 3.

In [ ]:
function plotly(data: object[], layout: object = {}) {
  const id = `plot-${crypto.randomUUID()}`;
  const html = `
<div id="${id}" style="width: 100%; max-width: 750px; height: 450px;"></div>
<script>
  (function () {
    const draw = () => Plotly.newPlot("${id}", ${JSON.stringify(data)}, ${JSON.stringify(layout)}, { responsive: true });
    if (window.Plotly) return draw();
    const script = document.createElement("script");
    script.src = "https://cdn.plot.ly/plotly-2.35.2.min.js";
    script.onload = draw;
    document.head.appendChild(script);
  })();
</script>`;
  Deno.jupyter.display({ "text/html": html }, { raw: true });
}

In [ ]:
const minWeight = Math.min(...weight);
const maxWeight = Math.max(...weight);
const lineX = [minWeight, maxWeight];
const lineY = lineX.map((w) => slope * w + intercept);

const data = [
  { x: weight, y: mpg, mode: "markers", type: "scatter", name: "Données", marker: { color: "#36a2eb", size: 6 } },
  { x: lineX, y: lineY, mode: "lines", type: "scatter", name: "Régression", line: { color: "#ff6384", width: 3 } },
];

const layout = {
  title: "Régression linéaire : MPG vs Poids",
  xaxis: { title: "Poids (lbs)" },
  yaxis: { title: "Consommation (MPG)" },
};

plotly(data, layout);

## 4.7 Calculer le MSE et le RMSE

In [ ]:
// Prédictions pour toutes les données
const predictions = weight.map((w) => slope * w + intercept);

// MSE : moyenne des carrés des écarts entre valeurs réelles et prédites
const mse = ss.mean(mpg.map((y, i) => (y - predictions[i]) ** 2));
const rmse = Math.sqrt(mse);

console.log(`MSE : ${mse.toFixed(4)}`);
console.log(`RMSE : ${rmse.toFixed(4)}`);
console.log(`R² : ${r2.toFixed(4)}`);